# Omni Engine: Sanity Gate and Ablation Matrix

This notebook runs the two confidence gates for the Omni Engine hybrid sparse neural system:

1. **Sanity gate**: a short training run proving the harness works, the loss descends, and Omni beats the frozen base.
2. **Ablation matrix**: flat vs graph vs hebbian vs full at equal parameters and equal steps.

Expected runtime on the free T4 GPU: roughly 30 minutes total.

When it finishes, copy the final comparison table and paste it back to your session.

In [ ]:
!git clone https://github.com/AbduljabbarBXR/Omni-Engine.git 2>/dev/null || true
%cd Omni-Engine
!git pull --ff-only 2>/dev/null || true
!git log --oneline -1

In [ ]:
!pip install -q transformers numpy
import torch
print("torch", torch.__version__)
print("cuda", torch.cuda.is_available())
print("gpu", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
import os
import subprocess
import sys

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

sanity_cmd = [
    sys.executable, "-m", "omni.train",
    "--steps", "60",
    "--batch-size", "8",
    "--seq-len", "256",
    "--base", "EleutherAI/pythia-160m-deduped",
    "--out", "runs/sanity",
    "--log-every", "10",
]
proc = subprocess.run(sanity_cmd, capture_output=True, text=True)
print(proc.stdout[-3000:])
if proc.returncode != 0:
    print(proc.stderr[-2000:])
print("SANITY_DONE", proc.returncode)

In [ ]:
sweep_cmd = [
    sys.executable, "stability_sweep.py",
    "--steps", "120",
    "--batch-size", "8",
    "--seq-len", "256",
    "--base", "EleutherAI/pythia-160m-deduped",
    "--out", "runs/sweep",
]
proc = subprocess.run(sweep_cmd, capture_output=True, text=True)
print(proc.stdout[-4000:])
if proc.returncode != 0:
    print(proc.stderr[-2000:])
print("SWEEP_DONE", proc.returncode)

In [ ]:
cmd = [
    sys.executable, "run_ablation.py",
    "--steps", "400",
    "--batch-size", "8",
    "--seq-len", "256",
    "--base", "EleutherAI/pythia-160m-deduped",
    "--out", "runs/ablation",
    "--eval-blocks", "100",
]
proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout[-6000:])
if proc.returncode != 0:
    print(proc.stderr[-2000:])
print("ABLATION_DONE", proc.returncode)

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    import shutil
    shutil.make_archive("/content/drive/MyDrive/omni_runs", "zip", "runs")
    print("runs zipped to MyDrive/omni_runs.zip")
except Exception as e:
    print("drive save skipped:", e)

# Paste back to your session

Copy the sanity gate output (loss curve and ppl) and the final ablation table. The delta column tells us everything:

* delta below 0 means Omni beats the frozen base.
* graph delta vs flat delta decides whether the learned topology is worth it.
* full delta decides whether Hebbian and graph compose.